# MedTrust-Net — Notebook A: Training the Hero Model

**Goal:** Train the full MedTrust-Net architecture (ResNet-50 + Confidence-Aware Attention Block + Hybrid Loss) on a 20k-image subset of CheXpert-v1.0-small, restricted to the 5 official CheXpert competition pathologies.

**Outputs (saved to Google Drive):**
- `best.pt` — best model by joint AUROC + ECE criterion
- `last.pt` — last epoch checkpoint
- `training_history.json` — per-epoch metrics
- `figures/` — class distribution, training curves, calibration plots

**Targets (from PPRS Section 2.5.3):**
- Macro-AUROC ≥ 0.85
- Macro-ECE ≤ 0.10
- Auto-stop when both held for 3 consecutive epochs

**Pathologies (CheXpert-5, Irvin et al. 2019):**
Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion


## 1. Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Install any missing packages (most are pre-installed on Colab)
!pip install -q torchmetrics==1.3.0


## 2. Configuration

In [ ]:
import os
from pathlib import Path

# --- Paths ---
DRIVE_ROOT       = Path('/content/drive/MyDrive')
ZIP_PATH         = DRIVE_ROOT / 'CheXpert-v1.0-small.zip'
EXTRACT_DIR      = Path('/content/chexpert')      # local Colab disk for fast I/O
RESULTS_DIR      = DRIVE_ROOT / 'medtrust_results'
CHECKPOINT_DIR   = RESULTS_DIR / 'checkpoints'
FIGURES_DIR      = RESULTS_DIR / 'figures'
METRICS_DIR      = RESULTS_DIR / 'metrics'

for d in [RESULTS_DIR, CHECKPOINT_DIR, FIGURES_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- CheXpert-5 pathologies (official competition subset) ---
PATHOLOGIES = [
    'Atelectasis',
    'Cardiomegaly',
    'Consolidation',
    'Edema',
    'Pleural Effusion',
]
NUM_CLASSES = len(PATHOLOGIES)

# --- Data sampling ---
TRAIN_SAMPLE_SIZE = 20000   # frontal X-rays only
VAL_SPLIT_RATIO   = 0.10    # carved out of the 20k

# --- Image preprocessing ---
IMAGE_SIZE = 224

# --- Training hyperparameters ---
BATCH_SIZE       = 32
NUM_WORKERS      = 2
STAGE1_EPOCHS    = 5        # frozen backbone
STAGE2_EPOCHS    = 25       # full fine-tuning
STAGE1_LR        = 1e-3
STAGE2_LR        = 1e-4
WEIGHT_DECAY     = 1e-4

# --- Hybrid loss weights ---
BETA_KL          = 1e-3     # KL regularization on CAB attention distributions
BETA_CAL         = 0.1      # calibration term (soft ECE surrogate)

# --- Auto-stop criterion ---
TARGET_AUROC     = 0.85
TARGET_ECE       = 0.10
PATIENCE_EPOCHS  = 3        # consecutive epochs hitting both targets

# --- Reproducibility ---
SEED = 42

import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Configuration loaded.")
print(f"Pathologies: {PATHOLOGIES}")
print(f"Train sample size: {TRAIN_SAMPLE_SIZE}")
print(f"Targets: AUROC ≥ {TARGET_AUROC}, ECE ≤ {TARGET_ECE}")


## 3. Extract Dataset

In [ ]:
import zipfile

if not (EXTRACT_DIR / 'train.csv').exists() and not (EXTRACT_DIR / 'CheXpert-v1.0-small' / 'train.csv').exists():
    print(f"Extracting {ZIP_PATH}...")
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

# Detect actual layout (some zips nest, some don't)
if (EXTRACT_DIR / 'train.csv').exists():
    DATA_ROOT = EXTRACT_DIR
elif (EXTRACT_DIR / 'CheXpert-v1.0-small' / 'train.csv').exists():
    DATA_ROOT = EXTRACT_DIR / 'CheXpert-v1.0-small'
else:
    raise FileNotFoundError("Could not locate train.csv after extraction")

print(f"Data root: {DATA_ROOT}")
print(f"Files in root: {sorted([p.name for p in DATA_ROOT.iterdir() if p.is_file()])[:10]}")


## 4. Load and Filter CheXpert Labels

In [ ]:
import pandas as pd

train_csv = pd.read_csv(DATA_ROOT / 'train.csv')
valid_csv = pd.read_csv(DATA_ROOT / 'valid.csv')

print(f"Raw train rows: {len(train_csv):,}")
print(f"Raw valid rows: {len(valid_csv):,}")
print(f"Columns: {list(train_csv.columns)[:8]}... ({len(train_csv.columns)} total)")


In [ ]:
# Filter to frontal views only (AP or PA), drop laterals
train_csv = train_csv[train_csv['Frontal/Lateral'] == 'Frontal'].copy()
valid_csv = valid_csv[valid_csv['Frontal/Lateral'] == 'Frontal'].copy()

print(f"Frontal-only train: {len(train_csv):,}")
print(f"Frontal-only valid: {len(valid_csv):,}")


In [ ]:
# Handle uncertainty labels (-1) using U-Ones strategy:
# Treat uncertain (-1) as positive (1). Irvin et al. 2019 found this works
# best for Atelectasis and Edema, and is the simplest defensible default.
# Missing values (NaN) are treated as negative (0).

def apply_u_ones(df, columns):
    df = df.copy()
    for col in columns:
        df[col] = df[col].fillna(0)
        df[col] = df[col].replace(-1, 1)
        df[col] = df[col].astype(int)
    return df

train_csv = apply_u_ones(train_csv, PATHOLOGIES)
valid_csv = apply_u_ones(valid_csv, PATHOLOGIES)

print("Applied U-Ones uncertainty handling.")
print("\nClass distribution (train, full set):")
for p in PATHOLOGIES:
    pos = train_csv[p].sum()
    pct = 100 * pos / len(train_csv)
    print(f"  {p:20s}: {pos:6,} positives ({pct:5.2f}%)")


## 5. Sample Training Subset & Train/Val Split

In [ ]:
# Stratified sampling: we want roughly preserved positive rates per class.
# Simplest robust approach: random sample, then verify distribution looks ok.
# CheXpert is multi-label so true stratification is hard; random sampling
# works well at 20k scale.

train_sampled = train_csv.sample(n=TRAIN_SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)

# Carve out internal validation split
val_size = int(TRAIN_SAMPLE_SIZE * VAL_SPLIT_RATIO)
val_df   = train_sampled.iloc[:val_size].reset_index(drop=True)
train_df = train_sampled.iloc[val_size:].reset_index(drop=True)

# Test set = official CheXpert valid.csv (radiologist-labeled, gold standard)
test_df = valid_csv.reset_index(drop=True)

print(f"Train: {len(train_df):,}")
print(f"Val:   {len(val_df):,}")
print(f"Test:  {len(test_df):,}  (official CheXpert valid set)")

print("\nClass distribution in our train split:")
for p in PATHOLOGIES:
    pos = train_df[p].sum()
    pct = 100 * pos / len(train_df)
    print(f"  {p:20s}: {pos:5,} ({pct:5.2f}%)")


In [ ]:
# --- Visualization: class distribution ---
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(9, 5))

pos_counts = [train_df[p].sum() for p in PATHOLOGIES]
neg_counts = [len(train_df) - c for c in pos_counts]
x = np.arange(len(PATHOLOGIES))
w = 0.35

bars1 = ax.bar(x - w/2, pos_counts, w, label='Positive', color='#d62728')
bars2 = ax.bar(x + w/2, neg_counts, w, label='Negative', color='#2ca02c')

ax.set_xticks(x)
ax.set_xticklabels(PATHOLOGIES, rotation=20, ha='right')
ax.set_ylabel('Number of images')
ax.set_title(f'CheXpert-5 Class Distribution (train split, n={len(train_df):,})')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Annotate positive percentages
for i, (p, n) in enumerate(zip(pos_counts, neg_counts)):
    pct = 100 * p / (p + n)
    ax.text(i - w/2, p + 100, f'{pct:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES_DIR / 'class_distribution.png'}")


## 6. Compute Class Weights (handles imbalance)

In [ ]:
# Per-class positive weights for weighted BCE: weight = N_neg / N_pos
# This is the standard pos_weight argument for BCEWithLogitsLoss in PyTorch.
pos_weights = []
for p in PATHOLOGIES:
    n_pos = train_df[p].sum()
    n_neg = len(train_df) - n_pos
    w = n_neg / max(n_pos, 1)
    pos_weights.append(w)

pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float32)

print("Per-class pos_weight (for weighted BCE):")
for p, w in zip(PATHOLOGIES, pos_weights):
    print(f"  {p:20s}: {w:6.2f}")


## 7. Dataset and DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class CheXpertDataset(Dataset):
    def __init__(self, df, data_root, pathologies, transform=None):
        self.df = df.reset_index(drop=True)
        self.data_root = Path(data_root)
        # CheXpert paths in the CSV start with 'CheXpert-v1.0-small/...'
        # We need to handle both layouts.
        self.path_prefix = self._detect_path_prefix()
        self.pathologies = pathologies
        self.transform = transform

    def _detect_path_prefix(self):
        # CSV path looks like: CheXpert-v1.0-small/train/patient00001/study1/view1_frontal.jpg
        # If our DATA_ROOT already points to CheXpert-v1.0-small/, strip the prefix.
        sample_path = self.df.iloc[0]['Path']
        if (self.data_root / sample_path).exists():
            return self.data_root
        # Try stripping the leading 'CheXpert-v1.0-small/' from CSV paths
        stripped = '/'.join(sample_path.split('/')[1:])
        if (self.data_root / stripped).exists():
            return 'STRIP'
        # Fall back to data_root.parent
        if (self.data_root.parent / sample_path).exists():
            return self.data_root.parent
        raise FileNotFoundError(f"Cannot locate image: {sample_path}")

    def _resolve_path(self, csv_path):
        if self.path_prefix == 'STRIP':
            return self.data_root / '/'.join(csv_path.split('/')[1:])
        return self.path_prefix / csv_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._resolve_path(row['Path'])
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        labels = torch.tensor(
            [row[p] for p in self.pathologies], dtype=torch.float32
        )
        return img, labels


# Standard ImageNet normalization (we use ImageNet-pretrained ResNet-50)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = CheXpertDataset(train_df, DATA_ROOT, PATHOLOGIES, train_transform)
val_dataset   = CheXpertDataset(val_df,   DATA_ROOT, PATHOLOGIES, eval_transform)
test_dataset  = CheXpertDataset(test_df,  DATA_ROOT, PATHOLOGIES, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# Sanity check
img, lbl = train_dataset[0]
print(f"Sample image shape: {img.shape}")
print(f"Sample labels: {lbl}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")


## 8. Confidence-Aware Attention Block (CAB)

The novel contribution of MedTrust-Net. Unlike standard attention which produces deterministic weights, CAB parameterizes attention as a Gaussian distribution with learned mean μ and log-variance log(σ²).

**Forward pass (training):** sample `attention = μ + σ·ε` via reparameterization trick, allowing gradients to flow.

**Forward pass (inference):** the mean μ becomes the **Diagnostic Attention Map (DAM)**, and σ becomes the **Confidence Reliability Map (CRM)** — high σ means the model is uncertain about that region.

**KL regularization:** we add KL divergence between the learned `N(μ, σ²)` and a standard Gaussian prior `N(0.5, 0.25)` (centered around mid-attention) to prevent the variance from collapsing to zero.

Architecture: spatial branch (7×7 conv) + channel branch (SE-style global pooling). The two are combined multiplicatively, following CBAM.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ConfidenceAwareAttentionBlock(nn.Module):
    """
    Confidence-Aware Attention Block (CAB).

    Replaces deterministic spatial+channel attention with a stochastic version
    that outputs both a mean attention map (DAM) and a variance map (CRM).
    """
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.in_channels = in_channels

        # --- Channel attention branch (SE-style, but outputs mu and log_var) ---
        self.channel_pool_avg = nn.AdaptiveAvgPool2d(1)
        self.channel_pool_max = nn.AdaptiveMaxPool2d(1)
        hidden = max(in_channels // reduction, 8)
        self.channel_mlp_mu = nn.Sequential(
            nn.Linear(in_channels * 2, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, in_channels),
        )
        self.channel_mlp_logvar = nn.Sequential(
            nn.Linear(in_channels * 2, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, in_channels),
        )

        # --- Spatial attention branch ---
        # Input to spatial conv: 2 channels (avg-pooled and max-pooled across channel dim)
        self.spatial_conv_mu = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.spatial_conv_logvar = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

        # Cached for KL computation and CRM extraction (only set during forward)
        self.last_spatial_mu = None
        self.last_spatial_logvar = None

    def reparameterize(self, mu, logvar):
        """Sample from N(mu, sigma^2) using the reparameterization trick."""
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu  # at inference, use the mean

    def forward(self, x):
        b, c, h, w = x.shape

        # ----- Channel attention -----
        avg_c = self.channel_pool_avg(x).view(b, c)
        max_c = self.channel_pool_max(x).view(b, c)
        ch_in = torch.cat([avg_c, max_c], dim=1)
        ch_mu     = self.channel_mlp_mu(ch_in)
        ch_logvar = self.channel_mlp_logvar(ch_in)
        ch_logvar = torch.clamp(ch_logvar, min=-10, max=2)
        ch_attn = self.reparameterize(ch_mu, ch_logvar)
        ch_attn = torch.sigmoid(ch_attn).view(b, c, 1, 1)

        # ----- Spatial attention -----
        avg_s = torch.mean(x, dim=1, keepdim=True)
        max_s = torch.max(x, dim=1, keepdim=True)[0]
        sp_in = torch.cat([avg_s, max_s], dim=1)
        sp_mu     = self.spatial_conv_mu(sp_in)
        sp_logvar = self.spatial_conv_logvar(sp_in)
        sp_logvar = torch.clamp(sp_logvar, min=-10, max=2)
        sp_attn = self.reparameterize(sp_mu, sp_logvar)
        sp_attn = torch.sigmoid(sp_attn)

        # Cache spatial mu/logvar for KL loss and for DAM/CRM extraction
        self.last_spatial_mu = sp_mu
        self.last_spatial_logvar = sp_logvar
        self.last_channel_mu = ch_mu
        self.last_channel_logvar = ch_logvar

        # Combined attention: channel * spatial, applied multiplicatively to x
        out = x * ch_attn * sp_attn
        return out

    def kl_divergence(self):
        """
        KL(N(mu, sigma^2) || N(0, 1)) summed over the cached spatial+channel mu/logvar.
        Standard VAE-style KL.
        """
        if self.last_spatial_mu is None:
            return torch.tensor(0.0)
        kl_sp = -0.5 * torch.mean(
            1 + self.last_spatial_logvar
              - self.last_spatial_mu.pow(2)
              - self.last_spatial_logvar.exp()
        )
        kl_ch = -0.5 * torch.mean(
            1 + self.last_channel_logvar
              - self.last_channel_mu.pow(2)
              - self.last_channel_logvar.exp()
        )
        return kl_sp + kl_ch


## 9. MedTrust-Net Architecture

ResNet-50 backbone (ImageNet pretrained) with a CAB inserted after each of the 4 residual stages. The final classifier uses global average pooling on the layer4 output. The layer4 CAB is the one whose spatial μ/σ become the DAM and CRM exposed to the user.


In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

class MedTrustNet(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        backbone = resnet50(weights=weights)

        # Stem
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )

        # Residual stages
        self.layer1 = backbone.layer1   # output: 256 channels
        self.layer2 = backbone.layer2   # output: 512 channels
        self.layer3 = backbone.layer3   # output: 1024 channels
        self.layer4 = backbone.layer4   # output: 2048 channels

        # CABs after each stage
        self.cab1 = ConfidenceAwareAttentionBlock(256)
        self.cab2 = ConfidenceAwareAttentionBlock(512)
        self.cab3 = ConfidenceAwareAttentionBlock(1024)
        self.cab4 = ConfidenceAwareAttentionBlock(2048)

        # Classifier head
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(2048, num_classes)

    def forward(self, x, return_maps=False):
        x = self.stem(x)
        x = self.layer1(x); x = self.cab1(x)
        x = self.layer2(x); x = self.cab2(x)
        x = self.layer3(x); x = self.cab3(x)
        x = self.layer4(x); x = self.cab4(x)

        feat = self.avgpool(x).flatten(1)
        feat = self.dropout(feat)
        logits = self.classifier(feat)

        if return_maps:
            # DAM = sigmoid(mu of layer4 spatial attention)
            # CRM = sigma of layer4 spatial attention (sqrt(exp(logvar)))
            dam = torch.sigmoid(self.cab4.last_spatial_mu)
            crm = torch.exp(0.5 * self.cab4.last_spatial_logvar)
            return logits, dam, crm
        return logits

    def total_kl_divergence(self):
        return (self.cab1.kl_divergence()
              + self.cab2.kl_divergence()
              + self.cab3.kl_divergence()
              + self.cab4.kl_divergence())

    def freeze_backbone(self):
        for p in self.stem.parameters(): p.requires_grad = False
        for p in self.layer1.parameters(): p.requires_grad = False
        for p in self.layer2.parameters(): p.requires_grad = False
        for p in self.layer3.parameters(): p.requires_grad = False
        for p in self.layer4.parameters(): p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.parameters():
            p.requires_grad = True


# Smoke test
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MedTrustNet(num_classes=NUM_CLASSES, pretrained=True).to(device)
dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
with torch.no_grad():
    out, dam, crm = model(dummy, return_maps=True)
print(f"Logits shape: {out.shape}")
print(f"DAM shape:    {dam.shape}")
print(f"CRM shape:    {crm.shape}")

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params:     {n_params/1e6:.2f}M")
print(f"Trainable params: {n_trainable/1e6:.2f}M")

# Estimate model size on disk (~ params * 4 bytes for float32)
print(f"Approx checkpoint size: {n_params * 4 / 1e6:.1f} MB")


## 10. Hybrid Loss Function

`L_total = L_BCE_weighted + β_kl · L_KL + β_cal · L_calibration`

- **L_BCE_weighted:** standard `BCEWithLogitsLoss` with per-class `pos_weight` to handle imbalance.
- **L_KL:** KL divergence on CAB attention distributions, weighted by `β_kl = 1e-3`.
- **L_calibration:** soft binned ECE surrogate — a differentiable approximation of Expected Calibration Error. Weighted by `β_cal = 0.1`.


In [ ]:
class SoftECELoss(nn.Module):
    """
    Differentiable soft-binned ECE surrogate.

    True ECE is non-differentiable because it uses hard binning. We use Gaussian
    soft-assignment to bins so gradients flow. This is a standard technique in
    calibration literature (e.g. Kumar et al., 2018 - 'Trainable Calibration Measures').
    """
    def __init__(self, n_bins=15):
        super().__init__()
        self.n_bins = n_bins
        bin_centers = torch.linspace(0.0, 1.0, n_bins)
        self.register_buffer('bin_centers', bin_centers)
        self.bin_width = 1.0 / n_bins

    def forward(self, logits, targets):
        # logits: (B, C), targets: (B, C) in {0, 1}
        probs = torch.sigmoid(logits)
        probs_flat = probs.reshape(-1)
        targets_flat = targets.reshape(-1)

        # Soft bin assignment via Gaussian kernel around each bin center
        sigma = self.bin_width
        diffs = probs_flat.unsqueeze(1) - self.bin_centers.unsqueeze(0)  # (N, n_bins)
        weights = torch.exp(-(diffs ** 2) / (2 * sigma ** 2))
        weights = weights / (weights.sum(dim=1, keepdim=True) + 1e-8)

        # Per-bin: avg confidence and avg accuracy
        bin_conf = (weights * probs_flat.unsqueeze(1)).sum(dim=0)
        bin_acc  = (weights * targets_flat.unsqueeze(1)).sum(dim=0)
        bin_count = weights.sum(dim=0) + 1e-8

        avg_conf = bin_conf / bin_count
        avg_acc  = bin_acc  / bin_count

        # Weighted absolute gap (this is the soft ECE)
        bin_weight = bin_count / bin_count.sum()
        ece = (bin_weight * torch.abs(avg_conf - avg_acc)).sum()
        return ece


class HybridLoss(nn.Module):
    def __init__(self, pos_weights, beta_kl=1e-3, beta_cal=0.1):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
        self.cal = SoftECELoss(n_bins=15)
        self.beta_kl = beta_kl
        self.beta_cal = beta_cal

    def forward(self, logits, targets, model):
        l_bce = self.bce(logits, targets)
        l_kl  = model.total_kl_divergence()
        l_cal = self.cal(logits, targets)
        total = l_bce + self.beta_kl * l_kl + self.beta_cal * l_cal
        return total, {
            'bce': l_bce.item(),
            'kl':  l_kl.item() if isinstance(l_kl, torch.Tensor) else l_kl,
            'cal': l_cal.item(),
            'total': total.item(),
        }


criterion = HybridLoss(
    pos_weights=pos_weights_tensor.to(device),
    beta_kl=BETA_KL,
    beta_cal=BETA_CAL,
)
print("Loss function ready.")


## 11. Evaluation Metrics (AUROC + ECE)

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

def compute_metrics(all_logits, all_labels, pathologies):
    """Return per-class AUROC, macro AUROC, and macro ECE (hard-binned, true ECE)."""
    probs  = torch.sigmoid(torch.from_numpy(all_logits)).numpy()
    labels = all_labels

    per_class_auroc = {}
    aurocs = []
    for i, p in enumerate(pathologies):
        if labels[:, i].sum() == 0 or labels[:, i].sum() == len(labels):
            per_class_auroc[p] = float('nan')
            continue
        a = roc_auc_score(labels[:, i], probs[:, i])
        per_class_auroc[p] = a
        aurocs.append(a)

    macro_auroc = float(np.mean(aurocs)) if aurocs else float('nan')

    # True (hard-binned) ECE per class, then macro-averaged
    def ece_one_class(p, y, n_bins=15):
        bin_edges = np.linspace(0, 1, n_bins + 1)
        ece = 0.0
        for j in range(n_bins):
            mask = (p > bin_edges[j]) & (p <= bin_edges[j+1])
            if mask.sum() == 0:
                continue
            avg_conf = p[mask].mean()
            avg_acc  = y[mask].mean()
            ece += (mask.sum() / len(p)) * abs(avg_conf - avg_acc)
        return ece

    eces = [ece_one_class(probs[:, i], labels[:, i]) for i in range(len(pathologies))]
    macro_ece = float(np.mean(eces))

    return {
        'per_class_auroc': per_class_auroc,
        'macro_auroc': macro_auroc,
        'macro_ece': macro_ece,
        'per_class_ece': dict(zip(pathologies, eces)),
    }


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        logits = model(imgs)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(lbls.numpy())
    return compute_metrics(np.concatenate(all_logits), np.concatenate(all_labels), PATHOLOGIES)


## 12. Training Loop with Auto-Stop

In [ ]:
import time, json
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_one_epoch(model, loader, optimizer, criterion, device, epoch):
    model.train()
    running = {'bce': 0, 'kl': 0, 'cal': 0, 'total': 0, 'n': 0}
    t0 = time.time()

    for batch_idx, (imgs, lbls) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(imgs)
        loss, parts = criterion(logits, lbls, model)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        bs = imgs.size(0)
        for k in ['bce', 'kl', 'cal', 'total']:
            running[k] += parts[k] * bs
        running['n'] += bs

        if (batch_idx + 1) % 50 == 0:
            print(f"  Epoch {epoch} | batch {batch_idx+1}/{len(loader)} | "
                  f"loss={parts['total']:.4f} (bce={parts['bce']:.4f} "
                  f"kl={parts['kl']:.4f} cal={parts['cal']:.4f})")

    elapsed = time.time() - t0
    n = running['n']
    return {k: running[k] / n for k in ['bce', 'kl', 'cal', 'total']} | {'epoch_time': elapsed}


def save_checkpoint(model, optimizer, epoch, metrics, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics,
        'pathologies': PATHOLOGIES,
    }, path)


In [ ]:
# ============ STAGE 1: Frozen backbone ============
print("=" * 60)
print("STAGE 1: Frozen backbone (training CAB + classifier only)")
print("=" * 60)

model.freeze_backbone()
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Stage 1 trainable params: {sum(p.numel() for p in trainable)/1e6:.2f}M")

optimizer = AdamW(trainable, lr=STAGE1_LR, weight_decay=WEIGHT_DECAY)

history = {'train': [], 'val': []}
best_score = -float('inf')   # we use AUROC - ECE as the joint score
hit_target_streak = 0
stage1_best_auroc = 0.0

for epoch in range(1, STAGE1_EPOCHS + 1):
    print(f"\n--- Stage 1 Epoch {epoch}/{STAGE1_EPOCHS} ---")
    train_stats = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch)
    val_metrics = evaluate(model, val_loader, device)

    print(f"  train loss: {train_stats['total']:.4f}")
    print(f"  val macro-AUROC: {val_metrics['macro_auroc']:.4f}")
    print(f"  val macro-ECE:   {val_metrics['macro_ece']:.4f}")

    history['train'].append({'epoch': epoch, 'stage': 1, **train_stats})
    history['val'].append({'epoch': epoch, 'stage': 1, **val_metrics})

    score = val_metrics['macro_auroc'] - val_metrics['macro_ece']
    if score > best_score:
        best_score = score
        save_checkpoint(model, optimizer, epoch, val_metrics, CHECKPOINT_DIR / 'best.pt')
        print(f"  -> new best (score={score:.4f}), saved best.pt")

    save_checkpoint(model, optimizer, epoch, val_metrics, CHECKPOINT_DIR / 'last.pt')


In [ ]:
# ============ STAGE 2: Full fine-tuning ============
print("\n" + "=" * 60)
print("STAGE 2: Full fine-tuning (unfrozen backbone)")
print("=" * 60)

model.unfreeze_backbone()
print(f"Stage 2 trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

optimizer = AdamW(model.parameters(), lr=STAGE2_LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=STAGE2_EPOCHS, eta_min=1e-6)

hit_target_streak = 0
final_epoch = STAGE1_EPOCHS

for epoch in range(STAGE1_EPOCHS + 1, STAGE1_EPOCHS + STAGE2_EPOCHS + 1):
    print(f"\n--- Stage 2 Epoch {epoch} (lr={optimizer.param_groups[0]['lr']:.2e}) ---")
    train_stats = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch)
    val_metrics = evaluate(model, val_loader, device)
    scheduler.step()

    print(f"  train loss: {train_stats['total']:.4f}")
    print(f"  val macro-AUROC: {val_metrics['macro_auroc']:.4f}")
    print(f"  val macro-ECE:   {val_metrics['macro_ece']:.4f}")

    history['train'].append({'epoch': epoch, 'stage': 2, **train_stats})
    history['val'].append({'epoch': epoch, 'stage': 2, **val_metrics})

    score = val_metrics['macro_auroc'] - val_metrics['macro_ece']
    if score > best_score:
        best_score = score
        save_checkpoint(model, optimizer, epoch, val_metrics, CHECKPOINT_DIR / 'best.pt')
        print(f"  -> new best (score={score:.4f}), saved best.pt")

    save_checkpoint(model, optimizer, epoch, val_metrics, CHECKPOINT_DIR / 'last.pt')

    # ----- Auto-stop check -----
    if (val_metrics['macro_auroc'] >= TARGET_AUROC
            and val_metrics['macro_ece'] <= TARGET_ECE):
        hit_target_streak += 1
        print(f"  Targets met! Streak: {hit_target_streak}/{PATIENCE_EPOCHS}")
        if hit_target_streak >= PATIENCE_EPOCHS:
            print(f"\n  *** Auto-stop triggered: targets held for {PATIENCE_EPOCHS} epochs ***")
            final_epoch = epoch
            break
    else:
        hit_target_streak = 0
    final_epoch = epoch

# Save full training history
with open(METRICS_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2, default=str)
print(f"\nTraining complete. History saved to {METRICS_DIR / 'training_history.json'}")


## 13. Final Test Set Evaluation

In [ ]:
# Load best checkpoint and evaluate on the official CheXpert valid set (our test)
ckpt = torch.load(CHECKPOINT_DIR / 'best.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']}")

test_metrics = evaluate(model, test_loader, device)
print("\n" + "=" * 50)
print("FINAL TEST SET METRICS (official CheXpert valid, n=234)")
print("=" * 50)
print(f"Macro AUROC: {test_metrics['macro_auroc']:.4f}")
print(f"Macro ECE:   {test_metrics['macro_ece']:.4f}")
print("\nPer-class AUROC:")
for p, a in test_metrics['per_class_auroc'].items():
    print(f"  {p:20s}: {a:.4f}")
print("\nPer-class ECE:")
for p, e in test_metrics['per_class_ece'].items():
    print(f"  {p:20s}: {e:.4f}")

# Save final metrics
with open(METRICS_DIR / 'medtrust_test_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)


## 14. Visualizations: Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs    = [h['epoch'] for h in history['val']]
val_auroc = [h['macro_auroc'] for h in history['val']]
val_ece   = [h['macro_ece'] for h in history['val']]
train_loss = [h['total'] for h in history['train']]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Train loss
axes[0].plot(epochs, train_loss, 'o-', color='#1f77b4')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Total loss')
axes[0].set_title('Training Loss')
axes[0].grid(alpha=0.3)
axes[0].axvline(STAGE1_EPOCHS + 0.5, color='gray', ls='--', alpha=0.5, label='Unfreeze')
axes[0].legend()

# AUROC
axes[1].plot(epochs, val_auroc, 'o-', color='#2ca02c', label='val macro-AUROC')
axes[1].axhline(TARGET_AUROC, color='red', ls='--', alpha=0.6, label=f'target {TARGET_AUROC}')
axes[1].axvline(STAGE1_EPOCHS + 0.5, color='gray', ls='--', alpha=0.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Macro AUROC')
axes[1].set_title('Validation AUROC')
axes[1].set_ylim(0.5, 1.0)
axes[1].legend()
axes[1].grid(alpha=0.3)

# ECE
axes[2].plot(epochs, val_ece, 'o-', color='#d62728', label='val macro-ECE')
axes[2].axhline(TARGET_ECE, color='red', ls='--', alpha=0.6, label=f'target {TARGET_ECE}')
axes[2].axvline(STAGE1_EPOCHS + 0.5, color='gray', ls='--', alpha=0.5)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Macro ECE')
axes[2].set_title('Validation ECE (lower is better)')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES_DIR / 'training_curves.png'}")


## 15. Sample DAM + CRM Visualization

In [ ]:
# Pick a few test images and visualize the DAM and CRM.
import numpy as np

model.eval()
n_samples = 4
fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))

with torch.no_grad():
    for i in range(n_samples):
        img, lbl = test_dataset[i]
        img_b = img.unsqueeze(0).to(device)
        logits, dam, crm = model(img_b, return_maps=True)
        probs = torch.sigmoid(logits)[0].cpu().numpy()

        # Resize maps to image size
        dam_up = F.interpolate(dam, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False)
        crm_up = F.interpolate(crm, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False)
        dam_np = dam_up[0, 0].cpu().numpy()
        crm_np = crm_up[0, 0].cpu().numpy()

        # Denormalize image for display
        img_disp = img.permute(1, 2, 0).numpy()
        img_disp = img_disp * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        img_disp = np.clip(img_disp, 0, 1)

        axes[i, 0].imshow(img_disp)
        axes[i, 0].set_title(f'Sample {i+1}: original X-ray')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(img_disp)
        axes[i, 1].imshow(dam_np, cmap='jet', alpha=0.5)
        top_idx = int(np.argmax(probs))
        axes[i, 1].set_title(f'DAM | top: {PATHOLOGIES[top_idx]} ({probs[top_idx]:.2f})')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(img_disp)
        axes[i, 2].imshow(crm_np, cmap='hot', alpha=0.5)
        axes[i, 2].set_title(f'CRM (uncertainty) | mean σ={crm_np.mean():.3f}')
        axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sample_dam_crm.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES_DIR / 'sample_dam_crm.png'}")


## 16. Summary

This notebook has:

1. Loaded CheXpert-v1.0-small, filtered to frontal views and the 5 official competition pathologies
2. Sampled 20,000 images and split into train (90%) / val (10%); used the official `valid.csv` as the held-out test set
3. Applied U-Ones for uncertain labels and computed per-class positive weights for imbalance handling
4. Implemented the **Confidence-Aware Attention Block (CAB)** as a stochastic attention layer with reparameterized spatial and channel branches
5. Built **MedTrust-Net** = ResNet-50 + 4 CABs + classifier, with a 2-stage freeze/unfreeze training schedule
6. Trained with the **Hybrid Loss** (weighted BCE + KL on attention distributions + soft-ECE calibration term)
7. Used **auto-stop** when val AUROC ≥ 0.85 AND val ECE ≤ 0.10 for 3 consecutive epochs
8. Saved best and last checkpoints to Drive
9. Evaluated on the official test set and produced training curves and qualitative DAM/CRM samples

**Next:** Notebook B trains the baselines (ResNet-50, MC Dropout, Deep Ensemble), runs the ablation study (CAB on/off × calibration loss on/off), and produces the full benchmarking visualizations.
